# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    build_batch_requests_openai, parse_batch_results_openai,
    run_judge_anthropic, run_judge_openai,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR, JUDGE_GPT4O_MINI_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.4 GB
Model: Qwen/Qwen3-4B-Thinking-2507


### 1.2 Load Model & Tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "22GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-4B-Thinking-2507
Layers: 36, hidden_dim: 2560


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [4]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, model, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

Starting fresh: 817 items


TruthfulQA knowledge check:   0%|          | 0/817 [00:00<?, ?it/s]

Done. Total: 817 | Passed: 338 | Failed: 479

Passed: 338 | Failed: 479


### 2.2 MMLU

In [5]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, model, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Starting fresh: 14042 items


MMLU knowledge check:   0%|          | 0/14042 [00:00<?, ?it/s]

Done. Total: 14042 | Passed: 4582 | Failed: 9460

Passed: 4582 | Failed: 9460


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [6]:
tqa_resp_df = run_factual_generation(
    tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

Starting fresh: 1155 rows across 3 configs
Config A: 338 rows to generate.


Config A:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Config B: 479 rows to generate.


Config B:   0%|          | 0/479 [00:00<?, ?it/s]

Config C: 338 rows to generate.


Config C:   0%|          | 0/338 [00:00<?, ?it/s]

Done. Total rows: 1155
config
B    479
A    338
C    338
config
B    479
A    338
C    338


#### 3.1.2 Claude Haiku Batch Judge

In [7]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

Saved 3465 requests → judge_truthfulQA_requests.jsonl
Submitted batch: msgbatch_018hwebjx5zyxv9XKdAMWem6
[msgbatch_018hwebjx5zyxv9XKdAMWem6] in_progress | succeeded=0  processing=3465  errored=0
Waiting 180s...
[msgbatch_018hwebjx5zyxv9XKdAMWem6] in_progress | succeeded=0  processing=3465  errored=0
Waiting 180s...
[msgbatch_018hwebjx5zyxv9XKdAMWem6] ended | succeeded=3465  processing=0  errored=0
Saved 1155 rows → judge_truthfulQA.csv


#### 3.1.3 GPT-4o-mini Batch Judge

In [11]:
tqa_gpt_df = run_judge_openai(
    tqa_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_TQA_PATH,
    state_path=JUDGE_GPT4O_MINI_TQA_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

Resuming: 1 parts, 0 completed
[Part 0 | batch_69e7557cbe6081908dd99790a7b7feda] completed | completed=3465  total=3465
Downloaded part 0 → judge_truthfulQA_results_part0.jsonl
Saved 1155 rows → judge_truthfulQA.csv


### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [12]:
mmlu_resp_df = run_factual_generation(
    mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(mmlu_resp_df["config"].value_counts().to_string())

Resuming: 13332 done, 5292 remaining
Config A: already complete, skipping.
Config B: 710 rows to generate.


Config B:   0%|          | 0/710 [00:00<?, ?it/s]

Config C: 4582 rows to generate.


Config C:   0%|          | 0/4582 [00:00<?, ?it/s]

Done. Total rows: 18624
config
B    9460
A    4582
C    4582
config
B    9460
A    4582
C    4582


#### 3.2.2 Claude Haiku Batch Judge

In [13]:
mmlu_haiku_df = run_judge_anthropic(
    mmlu_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

Saved 55872 requests → judge_mmlu_requests.jsonl
Submitted batch: msgbatch_01GbUPqpgcNVJWTFqyGgrHV5
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] in_progress | succeeded=0  processing=55872  errored=0
Waiting 180s...
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] in_progress | succeeded=0  processing=55872  errored=0
Waiting 180s...
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] in_progress | succeeded=0  processing=55872  errored=0
Waiting 180s...
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] in_progress | succeeded=0  processing=55872  errored=0
Waiting 180s...
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] in_progress | succeeded=0  processing=55872  errored=0
Waiting 180s...
[msgbatch_01GbUPqpgcNVJWTFqyGgrHV5] ended | succeeded=55872  processing=0  errored=0
Saved 18624 rows → judge_mmlu.csv


#### 3.2.3 GPT-4o-mini Batch Judge

In [14]:
mmlu_gpt_df = run_judge_openai(
    mmlu_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_MMLU_PATH,
    state_path=JUDGE_GPT4O_MINI_MMLU_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

Split into 9 parts
Submitted part 0: batch_69e8c9dd7d6881909f330f9e3b6d3e62
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] validating | completed=0  total=0
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 180s...
[Part 0 | batch_69e8c9dd7d6881909f330f9e3b6d3e62] in_progress | completed=0  total=6399
Waiting 

KeyboardInterrupt: 

### 3.3 Judge Result Compile

In [ ]:
if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
else:
    tqa_votes  = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_TQA_PATH, JUDGE_GPT4O_MINI_TQA_PATH,
        vote_cols=VOTE_COLS,
    )
    mmlu_votes = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_MMLU_PATH, JUDGE_GPT4O_MINI_MMLU_PATH,
        vote_cols=VOTE_COLS,
    )
    tqa_full  = build_full(tqa_votes,  tqa_resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
    tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

Saved tqa_full  (1185 rows) → truthfulQA_full.csv
Saved mmlu_full (22013 rows) → mmlu_full.csv

=== TruthfulQA — votes_correct distribution ===
config           A    B    C
votes_correct               
0               77  227  236
1               14   21   10
2               10   10    6
3               21   39    9
4               14   21    9
5               22   15   10
6              210  116   88

  Config A (truth)       — useful range: votes_correct ≥ 4
    >= 6/6 correct :   210 rows
    >= 5/6 correct :   232 rows
    >= 4/6 correct :   246 rows

  Config B (honest_mistake) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   227 rows
    <= 1/6 correct :   248 rows
    <= 2/6 correct :   258 rows

  Config C (deception) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   236 rows
    <= 1/6 correct :   246 rows
    <= 2/6 correct :   252 rows

=== MMLU — votes_correct distribution ===
config            A     B     C
votes_correct                  
0              2406

## Part 4: Scenario Response Generation

In [ ]:
scenario_resp_df = run_scenario_generation(
    deception_df, model, tokenizer, DEVICE,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

Starting fresh: 400 rows
400 rows to generate.


Scenario generation:   0%|          | 0/400 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generation complete.
Saved 200 pairs to scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  (Taking a deep breath, trying to sound sincere...   
1  (My face is flushed, and I'm trying to look ge...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  (Taking a deep breath, trying to sound sincere...  
1  (My f

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [ ]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

Saved probe_dataset (13996 rows) → probe_dataset.csv

Label distribution:
label
deception         5934
honest_mistake    4775
truth             3287

Domain distribution:
domain
factual    13596
social       400


### 5.2 Extract Activations

In [ ]:
activations_arr, labels_arr = run_extract_activations(
    probe_dataset, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
)
print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_gemma_4_e4b_it_activations ...
Download failed (LocalProtocolError: Illegal header value b'Bearer '). Running extraction ...
Starting fresh: 13996 samples


Extracting activations:   0%|          | 0/13996 [00:00<?, ?it/s]

Extracted and saved: activations (13996, 42, 2560)

Label counts: {'truth': 3287, 'honest_mistake': 4775, 'deception': 5934}


## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [ ]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

n_layers=42, representative layers: 25%→layer 9, 50%→layer 20, 75%→layer 31
k values to scan: [16, 32, 64, 128, 256, 512]
Fitting PCA with max_k=512 per layer, then slicing for each k.



Layer 9 (25%):
  k=  16 | var=0.525 | val_F1=0.625 | train_F1=0.628 | gap=0.003 | time=0.4s
  k=  32 | var=0.620 | val_F1=0.665 | train_F1=0.669 | gap=0.004 | time=1.0s
  k=  64 | var=0.723 | val_F1=0.704 | train_F1=0.713 | gap=0.009 | time=2.7s
  k= 128 | var=0.826 | val_F1=0.728 | train_F1=0.743 | gap=0.015 | time=13.3s
  k= 256 | var=0.914 | val_F1=0.747 | train_F1=0.775 | gap=0.028 | time=32.8s
  k= 512 | var=0.972 | val_F1=0.763 | train_F1=0.816 | gap=0.054 | time=64.5s

Layer 20 (50%):
  k=  16 | var=0.501 | val_F1=0.804 | train_F1=0.807 | gap=0.002 | time=0.4s
  k=  32 | var=0.581 | val_F1=0.826 | train_F1=0.827 | gap=0.001 | time=1.1s
  k=  64 | var=0.662 | val_F1=0.841 | train_F1=0.847 | gap=0.006 | time=3.6s
  k= 128 | var=0.746 | val_F1=0.848 | train_F1=0.861 | gap=0.013 | time=16.9s
  k= 256 | var=0.832 | val_F1=0.843 | train_F1=0.870 | gap=0.027 | time=32.7s
  k= 512 | var=0.910 | val_F1=0.840 | train_F1=0.888 | gap=0.048 | time=64.4s

Layer 31 (75%):
  k=  16 | var=0.608 

In [ ]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN,
)

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_gemma_4_e4b_it_activations ...


Download failed (Illegal header value b'Bearer ')
Running PCA (64 components) across 42 layers ...
Saved activations_pca64.npy       (13996, 42, 64)
Saved pca64_components.npy (42, 64, 2560)
Saved pca64_explained_variance.csv
Explained variance — mean: 0.678, min: 0.539, max: 0.820


### 6.2 Baseline: Binary Classifier

In [ ]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

binary probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C1.csv (42 rows)


binary probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C01.csv (42 rows)


In [ ]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

Saved auroc.png


### 6.3 Approach 1: Direct 3-Way LR Classifier

In [ ]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

3-way LR probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_3way_pca64.csv (42 rows)


In [ ]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [ ]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

3-way MLP probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_3way_mlp_pca64.csv (42 rows)


In [ ]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_lr_vs_mlp.png


### 6.5 Approach 3: 2-Stage LR Classifier

In [ ]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

cascaded LR probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_cascaded_lr.csv (42 rows)


In [ ]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.6 Approach 4: 2-Stage MLP Classifier

In [ ]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

cascaded MLP probe (layers):   0%|          | 0/42 [00:00<?, ?it/s]

Saved probe_results_cascaded_mlp.csv (42 rows)


In [ ]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_cascaded_lr_vs_mlp.png


## Part 7: Model Comparison

In [ ]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

Loaded all probe results.


In [ ]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

 layer  3-Way LR  3-Way MLP  Cascaded LR  Cascaded MLP
     0  0.613624   0.643539     0.592277      0.637760
     1  0.625425   0.652467     0.593325      0.650690
     2  0.647207   0.673673     0.626433      0.672628
     3  0.671865   0.693183     0.646459      0.694840
     4  0.688094   0.710438     0.655337      0.702879
     5  0.737088   0.752274     0.702654      0.749088
     6  0.731955   0.752549     0.700548      0.751309
     7  0.744391   0.753986     0.713414      0.756499
     8  0.702254   0.741974     0.664190      0.742046
     9  0.701952   0.733896     0.664245      0.739846
    10  0.696070   0.732360     0.668591      0.729115
    11  0.728284   0.750811     0.700843      0.749120
    12  0.741062   0.749947     0.711364      0.750356
    13  0.725805   0.744422     0.704843      0.745629
    14  0.774355   0.778124     0.766216      0.778407
    15  0.784059   0.782156     0.775757      0.779336
    16  0.792193   0.786046     0.789764      0.789108
    17  0.

In [ ]:
# ── Upload large files to HuggingFace Hub ────────────────────────────────────
# Temporary: paste your token here; will be moved to settings.py later
HF_TOKEN = ""  # paste your token

from huggingface_hub import HfApi

api = HfApi()
files_to_upload = [
    ACTIVATIONS_PATH,
    LABELS_PATH,
    ACTIVATIONS_PCA_PATH,
    PCA_COMPONENTS_PATH,
]

for path in files_to_upload:
    print(f"Uploading {path.name} ...")
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=HF_ACTIVATIONS_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    print(f"  done: {path.name}")

print("All uploads complete.")


Uploading activations.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: activations.npy
Uploading labels.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  done: activations_pca64.npy
Uploading pca64_components.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: pca64_components.npy
All uploads complete.
